In [122]:
import time
from google import genai
from google.genai import types
import random

client = genai.Client()

kitty_reference = types.VideoGenerationReferenceImage(
  image=kitty, # Generated separately with Nano Banana
  reference_type="asset"
)

puppy_reference = types.VideoGenerationReferenceImage(
  image=puppy, # Generated separately with Nano Banana
  reference_type="asset"
)


# 1) Generate the seed clip at 720p with the non-Fast model2
prompt = """First person POV
The player picks up a torch and walks towards the haunted castle gate.
"""
op = client.models.generate_videos(
    model="veo-3.1-fast-generate-preview", # reference images don't work with fast model
    prompt=prompt,
    image=puppy,
    config=types.GenerateVideosConfig(
      last_frame=kitty # Generated separately with Nano Banana
    ),
)
while not op.done:
    time.sleep(1)
    op = client.operations.get(op)

base_video = op.response.generated_videos[0]  # <-- Veo-generated Video
client.files.download(file=base_video.video)
base_video.video.save('base.mp4')



Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


### Working normal prompt

In [113]:
# 2) Extend it
extend_prompt = """First person POV
The user casts a spell and kicks the door open. Bats and ghosts fly out.

The hallway candles flicker on
"""

op = client.models.generate_videos(
    model="veo-3.1-fast-generate-preview",
    source=base_video,
    config=types.GenerateVideosConfig(
        reference_images=[kitty],
    ),
)
while not op.done:
    time.sleep(1)
    op = client.operations.get(op)

ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Unsupported video generation request. Please check the documentation for supported usage: https://ai.google.dev/gemini-api/docs/video', 'status': 'INVALID_ARGUMENT'}}

In [10]:
extend_video = op.response.generated_videos[0]  # <-- Veo-generated Video
client.files.download(file=extend_video.video)
extend_video.video.save('extend.mp4')

### Different params

In [ ]:
# 2) Extend it
extend_prompt = """First person POV
The user casts a spell to open the door open. Bats fly out. The hallway candles flicker on
"""

op = client.models.generate_videos(
    model="veo-3.1-generate-preview",
    source=types.GenerateVideosSourceDict(
            video=types.VideoDict(
                uri=base_video.video.uri
            ),
            prompt=extend_prompt
    ),
    config=types.GenerateVideosConfig(
        number_of_videos=1
    )
)
while not op.done:
    time.sleep(1)
    op = client.operations.get(op)

ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Unsupported video generation request. Please check the documentation for supported usage: https://ai.google.dev/gemini-api/docs/video', 'status': 'INVALID_ARGUMENT'}}

In [24]:
extend_video = op.response.generated_videos[0]  # <-- Veo-generated Video
client.files.download(file=extend_video.video)
extend_video.video.save('extend_diff.mp4')

In [44]:
def generate_base_video(prompt):
    op = client.models.generate_videos(
        model="veo-3.1-fast-generate-preview",
        prompt=prompt
    )
    return op

def extend_video_without_prompt(base_video):
    op = client.models.generate_videos(
        model="veo-3.1-fast-generate-preview",
        source=base_video
    )
    return op

def extend_video_with_prompt(base_video, prompt):
    op = client.models.generate_videos(
        model="veo-3.1-generate-preview",
        source=types.GenerateVideosSourceDict(
                video=types.VideoDict(
                    uri=base_video.video.uri
                ),
                prompt=prompt
        ),
        config=types.GenerateVideosConfig(
            number_of_videos=1
        )
    )
    return op

def download_video(op, filename='extend_diff.mp4'):
    """
    Download the video from the operation
    returns:
        extend_video: The generated video
    """
    while not op.done:
        time.sleep(1)
        op = client.operations.get(op)
        
    extend_video = op.response.generated_videos[0]  # <-- Veo-generated Video
    client.files.download(file=extend_video.video)
    extend_video.video.save(filename)
    return extend_video


In [43]:
basis = base_video

for i in range(10):
    try:
        op = extend_video_without_prompt(basis)
        basis = download_video(op)
    except Exception as e:
        print(e)
        print("Retrying...")
        time.sleep(5)
        continue

In [40]:
download_video(op)

GeneratedVideo(
  video=Video(
    uri='https://generativelanguage.googleapis.com/v1beta/files/znlzvd1h89jk:download?alt=media',
    video_bytes=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free\x01:\x16Qmdat\x00\x00\x002\x06\x05.\xdcE\xe9\xbd\xe6\xd9H\xb7\x96,\xd8 \xd9#\xee\xefx264 - core 155 r2901 7d0f...'
  )
)

In [47]:
op = generate_base_video("POV walking through San Francisco")
basis = download_video(op, filename='test.mp4')

## Nano banana tests

In [ ]:
from google import genai
from google.genai import types
from PIL import Image
from io import BytesIO

prompt = "Panning wide shot of a calico kitten sleeping in the sunshine"

# Step 1: Generate an image with Nano Banana.
image = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents=[prompt],
)

for part in image.candidates[0].content.parts:
    if part.text is not None:
        print(part.text)
    elif part.inline_data is not None:
        saved_image = Image.open(BytesIO(part.inline_data.data))
        saved_image.save("generated_image.png")

Here you go! 


In [94]:
# Step 2: Generate video with Veo 3.1 using the image.
operation = client.models.generate_videos(
    model="veo-3.1-fast-generate-preview",
    prompt=prompt,
    image=types.Image(
        image_bytes=image.candidates[0].content.parts[1].inline_data.data,
        mime_type="image/png"
    )
)

In [99]:
download_video(operation)

GeneratedVideo(
  video=Video(
    uri='https://generativelanguage.googleapis.com/v1beta/files/3o6oyrsv6ki2:download?alt=media',
    video_bytes=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free\x00\x0f$\xd5mdat\x00\x00\x002\x06\x05.\xdcE\xe9\xbd\xe6\xd9H\xb7\x96,\xd8 \xd9#\xee\xefx264 - core 155 r2901 7d0f...'
  )
)

In [ ]:
def create_image_from_prompt(prompt) -> types.Image:
    response = client.models.generate_content(
        model="gemini-2.5-flash-image",
        contents=[prompt],
        config=types.GenerateContentConfig(
            image_config=types.ImageConfig(
                aspect_ratio="16:9",
            )
        )
    )
    for part in response.candidates[0].content.parts:
        if part.text is not None:
            print(part.text)
        elif part.inline_data is not None:
            return types.Image(
                image_bytes=part.inline_data.data,
                mime_type="image/png"
            )
    raise Exception("No image found in the response")

def show_image(image: types.Image):
    image = Image.open(BytesIO(image.image_bytes))
    image.show()


In [100]:
kitty = create_image_from_prompt("A calico kitten sleeping in the sunshine")
show_image(kitty)


Here is a calico kitten sleeping in the sunshine for you: 


In [121]:
puppy = create_image_from_prompt("A puppy playing with a ball")
show_image(puppy)


Here is an image of a puppy playing with a ball: 
